# Experimento 7: XLM-RoBERTa-base para Relation Extraction

**Objetivo:** Comparar XLM-RoBERTa-base con PubMedBERT usando exactamente la misma configuracion que el Experimento 1.

**Hipotesis:** XLM-RoBERTa no es especifico de biomedicina, pero es la arquitectura que uso el equipo ganador de BioNNE 2024 (fulstock). Su preentrenamiento masivo en 100 idiomas con RoBERTa puede compensar la falta de especializacion biomedica.

**Nota tecnica:** OpenNRE esta disenado para BERT (WordPiece, [CLS]/[SEP], [unused0]-[unused3]). XLM-RoBERTa usa SentencePiece y tokens <s>/<\/s>, y no tiene [unused] en el vocabulario. Se necesitan parches adicionales.

**Resultados anteriores:**
- Baseline (bert-multilingual, 10ep): Macro F1 = 0.6944
- Exp1: PubMedBERT (10ep): Macro F1 = 0.7754
- Exp2: PubMedBERT + typed markers (15ep): Macro F1 = 0.8078
- Exp3: PubMedBERT + typed markers + neg_ratio 1:1 (15ep): Macro F1 = 0.8430
- Exp6: BioLinkBERT-base (10ep): Macro F1 = 0.7576

**Modelo a probar:** `xlm-roberta-base`

## 1. Setup e Instalacion

In [ ]:
!pip install git+https://github.com/thunlp/OpenNRE.git
!pip install torch transformers nltk pandas scikit-learn matplotlib seaborn
!pip install sentencepiece

In [ ]:
import json
import importlib
import time
import logging
import os
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuracion del Experimento

In [ ]:
MODEL_NAME = "xlm-roberta-base"
EXPERIMENT_NAME = "xlm_roberta_base"

MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 10
WARMUP_STEPS = 300
SEED = 42
NEG_RATIO = 3

DATA_DIR = Path("/kaggle/input/datasets/lucaespernjj/bionner-data")
TRAIN_DATA = DATA_DIR / "eng_train.txt"
DEV_DATA = DATA_DIR / "eng_dev.txt"
REL2ID_PATH = DATA_DIR / "rel2id.json"

OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUTPUT_DIR / f"eng_{EXPERIMENT_NAME}.pth.tar"
PRED_PATH = OUTPUT_DIR / f"eng_pred_{EXPERIMENT_NAME}.tsv"

RELATION_TYPES = [
    "ABBREVIATION", "ALTERNATIVE_NAME", "SUBCLASS_OF", "PART_OF",
    "TREATED_USING", "ORIGINS_FROM", "TO_DETECT_OR_STUDY", "AFFECTS",
    "HAS_CAUSE", "APPLIED_TO", "USED_IN", "ASSOCIATED_WITH",
    "PHYSIOLOGY_OF", "FINDING_OF", "no_relation",
]

with open(REL2ID_PATH) as f:
    rel2id = json.load(f)

print(f"Experimento: {EXPERIMENT_NAME}")
print(f"Modelo: {MODEL_NAME}")

## 3. Verificar Datos

In [ ]:
assert TRAIN_DATA.exists() and DEV_DATA.exists() and REL2ID_PATH.exists()

with open(TRAIN_DATA) as f:
    n_train = sum(1 for line in f if line.strip())
with open(DEV_DATA) as f:
    n_dev = sum(1 for line in f if line.strip())

print(f"Train: {n_train} | Dev: {n_dev}")

train_instances = []
with open(TRAIN_DATA, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            train_instances.append(json.loads(line))

rel_counts = Counter(inst["relation"] for inst in train_instances)
for rel, count in rel_counts.most_common():
    print(f"  {rel:<25} {count:>6} ({100*count/len(train_instances):5.1f}%)")

## 4. Parchear OpenNRE para XLM-RoBERTa

**Parche 1:** AdamW de torch (igual que todos los experimentos).

**Parche 2:** Compatibilidad con XLM-RoBERTa. OpenNRE hardcodea BertModel + BertTokenizer + [CLS]/[SEP] + [unused0-3]. Para XLM-RoBERTa hay que:
- Usar AutoModel + AutoTokenizer(use_fast=False) → necesario para SentencePiece
- Usar tokenizer.cls_token / sep_token → para XLM-RoBERTa son `<s>` y `</s>`
- Añadir [unused0-5] al vocabulario y redimensionar embeddings
- **Inicializar nuevos embeddings con la media** (no random) → evita que el modelo colapse a predecir siempre `no_relation`
- Usar pad_token_id del tokenizador (XLM-RoBERTa usa 1, no 0)

In [ ]:
# ============================================================
# PARCHE 1: AdamW
# ============================================================
import opennre.framework.sentence_re as sre

sre_path = Path(sre.__file__)
sre_text = sre_path.read_text(encoding="utf-8")
sre_text = sre_text.replace("from transformers import AdamW", "from torch.optim import AdamW")
sre_text = sre_text.replace(
    "self.optimizer = AdamW(grouped_params, correct_bias=False)",
    "self.optimizer = AdamW(grouped_params)"
)
sre_path.write_text(sre_text, encoding="utf-8")
importlib.reload(sre)
opennre.framework.SentenceRE = sre.SentenceRE
print("Parche 1 (AdamW): OK")

# ============================================================
# PARCHE 2: Compatibilidad XLM-RoBERTa
# ============================================================
import opennre.encoder.bert_encoder as be

be_path = Path(be.__file__)
be_text = be_path.read_text(encoding="utf-8")

# 2a. BertModel + BertTokenizer -> AutoModel + AutoTokenizer
be_text = be_text.replace(
    "from transformers import BertModel, BertTokenizer",
    "from transformers import AutoModel, AutoTokenizer"
)
be_text = be_text.replace(
    "self.bert = BertModel.from_pretrained(pretrain_path)",
    "self.bert = AutoModel.from_pretrained(pretrain_path)"
)
be_text = be_text.replace(
    "self.tokenizer = BertTokenizer.from_pretrained(pretrain_path)",
    "self.tokenizer = AutoTokenizer.from_pretrained(pretrain_path, use_fast=False)"
)

# 2b. Anadir [unused0-5] al vocabulario e inicializar con la MEDIA de embeddings existentes.
# CRITICO: inicializar con random causa que el modelo prediga siempre no_relation.
# Inicializar con la media da un punto de partida util y el modelo aprende normalmente.
RESIZE_PATCH = (
    "        # Compatibilidad XLM-RoBERTa: anadir tokens de entidad\n"
    "        _entity_tokens = ['[unused0]', '[unused1]', '[unused2]',\n"
    "                          '[unused3]', '[unused4]', '[unused5]']\n"
    "        _new_tokens = [t for t in _entity_tokens if t not in self.tokenizer.get_vocab()]\n"
    "        if _new_tokens:\n"
    "            self.tokenizer.add_tokens(_new_tokens)\n"
    "            self.bert.resize_token_embeddings(len(self.tokenizer))\n"
    "            import torch as _torch\n"
    "            with _torch.no_grad():\n"
    "                n_new = len(_new_tokens)\n"
    "                emb = self.bert.get_input_embeddings().weight\n"
    "                avg = emb[:-n_new].mean(dim=0)\n"
    "                emb[-n_new:] = avg.unsqueeze(0).repeat(n_new, 1)\n"
)
be_text = be_text.replace(
    "        self.linear = nn.Linear(self.hidden_size, self.hidden_size)\n",
    RESIZE_PATCH + "        self.linear = nn.Linear(self.hidden_size, self.hidden_size)\n"
)

# 2c. [CLS]/[SEP] hardcodeados -> tokens reales del tokenizador
be_text = be_text.replace(
    "re_tokens = ['[CLS]'] + sent0 + ent0 + sent1 + ent1 + sent2 + ['[SEP]']",
    "re_tokens = [self.tokenizer.cls_token] + sent0 + ent0 + sent1 + ent1 + sent2 + [self.tokenizer.sep_token]"
)

# 2d. Padding: BERT=0, XLM-RoBERTa=1
be_text = be_text.replace(
    "indexed_tokens.append(0)  # 0 is id for [PAD]",
    "indexed_tokens.append(self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0)"
)

be_path.write_text(be_text, encoding="utf-8")
importlib.reload(be)
opennre.encoder.BERTEntityEncoder = be.BERTEntityEncoder
print("Parche 2 (XLM-RoBERTa): OK")
print("Todos los parches aplicados.")

## 5. Crear y Entrenar el Modelo

In [ ]:
print("=" * 60)
print(f"ENTRENAMIENTO: {EXPERIMENT_NAME}")
print(f"Modelo: {MODEL_NAME}")
print(f"Epochs: {EPOCHS} | LR: {LEARNING_RATE} | Batch: {BATCH_SIZE}")
print("=" * 60)

encoder = opennre.encoder.BERTEntityEncoder(
    max_length=MAX_LENGTH,
    pretrain_path=MODEL_NAME,
)

model = opennre.model.SoftmaxNN(
    sentence_encoder=encoder,
    num_class=len(rel2id),
    rel2id=rel2id,
)

framework = opennre.framework.SentenceRE(
    model=model,
    train_path=str(TRAIN_DATA),
    val_path=str(DEV_DATA),
    test_path=str(DEV_DATA),
    ckpt=str(CKPT_PATH),
    batch_size=BATCH_SIZE,
    max_epoch=EPOCHS,
    lr=LEARNING_RATE,
    opt="adamw",
    warmup_step=WARMUP_STEPS,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nParametros: {n_params:,}")
print("Listo para entrenar.")

In [ ]:
start_time = time.time()
framework.train_model(metric="micro_f1")
elapsed = time.time() - start_time
print(f"\nEntrenamiento completado en {elapsed/60:.1f} minutos")

## 6. Prediccion en Dev

In [ ]:
encoder_pred = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL_NAME)
model_pred = opennre.model.SoftmaxNN(sentence_encoder=encoder_pred, num_class=len(rel2id), rel2id=rel2id)

ckpt = torch.load(str(CKPT_PATH), map_location="cpu")
model_pred.load_state_dict(ckpt["state_dict"])
if torch.cuda.is_available():
    model_pred = model_pred.cuda()
model_pred.eval()
print(f"Modelo cargado desde: {CKPT_PATH}")

In [ ]:
dev_instances = []
with open(DEV_DATA, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            dev_instances.append(json.loads(line))

print(f"Prediciendo {len(dev_instances)} instancias...")

rows = []
for inst in dev_instances:
    pred_rel, score = model_pred.infer({
        "text": inst["text"],
        "h": {"pos": inst["h"]["pos"]},
        "t": {"pos": inst["t"]["pos"]},
    })
    rows.append({
        "document_id": inst["doc_id"], "relation": pred_rel, "score": score,
        "gold": inst["relation"],
        "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
        "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"],
    })

pred_df = pd.DataFrame(rows)
pred_export = pred_df[["document_id", "relation", "head_text", "head_span",
                        "head_type", "tail_text", "tail_span", "tail_type"]].copy()
pred_export = pred_export[pred_export["relation"] != "no_relation"]
pred_export.to_csv(PRED_PATH, sep="\t", index=False)
print(f"Predicciones guardadas en: {PRED_PATH}")

## 7. Evaluacion y Comparacion

In [ ]:
all_relations = sorted(rel2id.keys())
gold_labels = [inst["relation"] for inst in dev_instances]
pred_labels = [row["relation"] for row in rows]

correct = sum(1 for g, p in zip(gold_labels, pred_labels) if g == p)
total = len(gold_labels)
accuracy = correct / total

gold_counts = Counter(gold_labels)
pred_counts = Counter(pred_labels)
tp_counts = Counter()
for g, p in zip(gold_labels, pred_labels):
    if g == p:
        tp_counts[g] += 1

results = {}
print(f"{'Relacion':<25} {'P':>8} {'R':>8} {'F1':>8} {'Soporte':>8}")
print("-" * 60)

f1_scores = []
for rel in all_relations:
    tp = tp_counts.get(rel, 0)
    pred_total = pred_counts.get(rel, 0)
    gold_total = gold_counts.get(rel, 0)
    p = tp / pred_total if pred_total else 0
    r = tp / gold_total if gold_total else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0
    results[rel] = {"precision": p, "recall": r, "f1": f1, "support": gold_total}
    if gold_total > 0:
        f1_scores.append(f1)
        print(f"{rel:<25} {p:>8.4f} {r:>8.4f} {f1:>8.4f} {gold_total:>8}")

macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0
print("-" * 60)
print(f"\nAccuracy: {accuracy:.4f} | Macro F1: {macro_f1:.4f}")

print(f"\n{'='*65}")
print(f"{'Experimento':<45} {'Macro F1':>10} {'Diff':>10}")
print(f"{'-'*65}")
print(f"{'Baseline (bert-multi, 10ep)':<45} {'0.6944':>10} {'---':>10}")
print(f"{'Exp1: PubMedBERT (10ep)':<45} {'0.7754':>10} {'+0.0810':>10}")
print(f"{'Exp2: PubMedBERT + typed markers (15ep)':<45} {'0.8078':>10} {'+0.1134':>10}")
print(f"{'Exp3: PubMedBERT + typed + neg 1:1 (15ep)':<45} {'0.8430':>10} {'+0.1486':>10}")
print(f"{'Exp6: BioLinkBERT-base (10ep)':<45} {'0.7576':>10} {'+0.0632':>10}")
print(f"{'Exp7: XLM-RoBERTa-base (10ep)':<45} {macro_f1:>10.4f} {macro_f1 - 0.6944:>+10.4f}")
print(f"{'='*65}")
print(f"XLM-RoBERTa vs PubMedBERT: {macro_f1 - 0.7754:+.4f}")

## 8. Analisis de Errores

In [ ]:
confusion_pairs = Counter()
for g, p in zip(gold_labels, pred_labels):
    if g != p:
        confusion_pairs[(g, p)] += 1

total_errors = sum(confusion_pairs.values())
print(f"Total errores: {total_errors} / {total} ({100*total_errors/total:.1f}%)")

exp1_errors = {
    ("ALTERNATIVE_NAME", "no_relation"): 53,
    ("HAS_CAUSE", "no_relation"): 53,
    ("PART_OF", "SUBCLASS_OF"): 51,
    ("AFFECTS", "no_relation"): 36,
    ("SUBCLASS_OF", "HAS_CAUSE"): 26,
}
print(f"\n{'Confusion':<50} {'Exp1':>6} {'Exp7':>6} {'Cambio':>8}")
print("-" * 75)
for (g, p), prev_count in sorted(exp1_errors.items(), key=lambda x: -x[1]):
    curr_count = confusion_pairs.get((g, p), 0)
    diff = curr_count - prev_count
    print(f"{g} -> {p:<30} {prev_count:>6} {curr_count:>6} {diff:>+8}")

print(f"\nTop 10 confusiones (XLM-RoBERTa):")
for (g, p), count in confusion_pairs.most_common(10):
    print(f"{g:<25} {p:<25} {count:>6}")

In [ ]:
pubmedbert_f1 = {
    "ABBREVIATION": 0.9200, "AFFECTS": 0.8796, "ALTERNATIVE_NAME": 0.3465,
    "APPLIED_TO": 0.5743, "ASSOCIATED_WITH": 0.8107, "FINDING_OF": 0.6821,
    "HAS_CAUSE": 0.8179, "ORIGINS_FROM": 0.8571, "PART_OF": 0.7862,
    "PHYSIOLOGY_OF": 0.8571, "SUBCLASS_OF": 0.8728, "TO_DETECT_OR_STUDY": 0.8072,
    "TREATED_USING": 0.9024, "USED_IN": 0.7421,
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Experimento 7: XLM-RoBERTa-base vs PubMedBERT", fontsize=14, fontweight="bold")

rels_plot = [r for r in all_relations if results.get(r, {}).get("support", 0) > 0]
f1_pubmed = [pubmedbert_f1.get(r, 0) for r in rels_plot]
f1_xlm = [results[r]["f1"] for r in rels_plot]

x = np.arange(len(rels_plot))
width = 0.35
axes[0].bar(x - width/2, f1_pubmed, width, label='PubMedBERT (0.7754)', color='#3498db', alpha=0.8)
axes[0].bar(x + width/2, f1_xlm, width, label=f'XLM-RoBERTa ({macro_f1:.4f})', color='#9b59b6', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(rels_plot, rotation=45, ha='right', fontsize=8)
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].set_title('F1 por relacion')

diffs = [f1_xlm[i] - f1_pubmed[i] for i in range(len(rels_plot))]
axes[1].barh(rels_plot, diffs, color=['#2ecc71' if d > 0 else '#e74c3c' for d in diffs])
axes[1].set_xlabel('Cambio en F1 (XLM-RoBERTa - PubMedBERT)')
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_title('Impacto de cambiar backbone')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / f"comparacion_{EXPERIMENT_NAME}.png"), dpi=150, bbox_inches='tight')
plt.show()

## 9. Guardar Resultados

In [ ]:
experiment_results = {
    "experiment": EXPERIMENT_NAME, "model": MODEL_NAME,
    "hyperparameters": {
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "epochs": EPOCHS,
        "warmup_steps": WARMUP_STEPS, "neg_ratio": NEG_RATIO,
    },
    "results": {"accuracy": accuracy, "macro_f1": macro_f1, "per_relation": results},
    "comparison": {
        "baseline": 0.6944, "pubmedbert": 0.7754,
        "biolinkbert_base": 0.7576, "xlm_roberta_base": macro_f1,
        "diff_vs_pubmedbert": macro_f1 - 0.7754,
    },
    "error_analysis": {
        "total_errors": total_errors,
        "top_confusions": [{"gold": g, "pred": p, "count": c} for (g, p), c in confusion_pairs.most_common(10)],
    },
}

results_path = OUTPUT_DIR / f"results_{EXPERIMENT_NAME}.json"
with open(results_path, "w") as f:
    json.dump(experiment_results, f, indent=2)

print(f"Macro F1: {macro_f1:.4f}")
print(f"vs PubMedBERT: {macro_f1 - 0.7754:+.4f}")
print(f"vs Exp3 (best): {macro_f1 - 0.8430:+.4f}")